## 4.1 Pytorch 搭建MLP - 搭建流程

在前面的章节中，我们已经学习了：
* 神经网络结构（MLP）
* 矩阵计算（XW^T + b）
* 激活函数（ReLU / Sigmoid / Softmax）
* 参数初始化

现在我们开始进入 真正的模型构建阶段：
使用 PyTorch 构建一个基础的 MLP 神经网络。

这一节的重点是先理解：
一个神经网络模型在 PyTorch 中是如何一步一步搭建出来的。

#### 1. PyTorch 搭建 MLP 的整体流程
在 PyTorch 中构建神经网络通常遵循下面的流程：

##### 1.1 导入 PyTorch 相关库
其中：
* torch → PyTorch 核心库
* torch.nn → 神经网络模块（layers / activation / loss 等）

In [1]:
import torch
import torch.nn as nn

##### 1.2 定义模型类（继承 nn.Module）
在 PyTorch 中，所有神经网络模型都需要继承：
`nn.Module`

例如：

`class MLP(nn.Module):`

这一步的含义是：

我们正在定义一个新的神经网络模型。

##### 1.3 在 `__init__()` 中定义网络结构
`__init__()` 函数负责定义：
* 网络层（Linear）
* 激活函数（ReLU）
* 网络结构

例如：
* 输入层
* 隐藏层
* 输出层

这些内容决定了：
`模型长什么样（Model Architecture）`

例如：
```
Linear
ReLU
Linear
```
这些层通常写成：

`self.layer_name`

这样 PyTorch 才能自动管理这些层的参数。

##### 1.4 在 forward() 中定义前向传播
forward() 函数负责定义：

`输入数据如何在网络中进行计算`

也就是数据的流动路径：

`输入 → 第一层 → 激活函数 → 第二层 → 输出`

例如：

`x → layer1 → relu → layer2 → output`

当我们调用：

`model(X)`

PyTorch 会自动执行：

`forward(X)`

因此：

forward() 定义的是模型的计算逻辑（computational graph）。

##### 1.5 创建模型实例
当模型类定义完成后，需要创建一个模型对象：

`model = MLP()`

此时：
* 网络结构已经创建
* 所有参数（权重和偏置）已经初始化
    * 模型会自动初始化
    * 可以手动干预
* 模型可以接收输入数据进行前向计算

##### 本节核心总结 🧠
在 PyTorch 中构建一个 MLP 模型的核心步骤只有四个：
```
导入库
↓
定义模型类（nn.Module）
↓
在 __init__() 中定义网络结构
↓
在 forward() 中定义前向传播
↓
创建模型实例
```

🧠 结构在 `__init__`，计算在 `forward`
```
MLP模型
│
├── __init__()
│      ├── Linear
│      ├── ReLU
│      └── Linear
│
└── forward()
       ├── layer1
       ├── relu
       └── layer2
```

#### 2. PyTorch 实现 MLP（模型结构定义）
在 PyTorch 中，神经网络模型通常通过继承：

`nn.Module`

`nn.Module` 是 PyTorch 所有神经网络模型的基类。

我们先设计一个简单的 MLP：
```
输入：
3 个特征
隐藏层：
4 个神经元
输出：
1 个神经元
```

In [3]:
class MLP(nn.Module):
    # 定义MLP的结构
    def __init__(self):
        # 调用父类的构造函数, 目的是初始化父类的属性和方法
        super().__init__()
        # 输入层：3个神经元（3个特征）， 输出4个神经元（4个特征）
        self.layer_1 = nn.Linear(in_features = 3, out_features = 4)
        # 激活函数
        self.relu = nn.ReLU()
        # 隐藏层：接受上一层的4个特征（4个神经元），输出1个特征（1个神经元）
        self.layer_2 = nn.Linear(in_features = 4, out_features = 1)
    
    # 定义MLP的计算图（forward）
    def forward(self, x):
        # 计算图：输入数据x -> layer_1 -> relu -> layer_2 -> 输出
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

##### 2.1 self 表示什么意思？
self 是 Python 面向对象编程中的概念。

**Java 是 强类型语言，类结构在编译时就确定。**

例如：
``` java
class MLP {

    Linear layer1;
} 
```
这里 layer1是 类的成员变量（field）。

Java 编译器会自动帮你做：

`this.layer1`

也就是说：

**在 Java 中，layer1 实际上就是 this.layer1,只是 Java 允许省略 this。**

**Python 的类结构更加动态。**

**在 Python 中：**
* **类里只定义 方法，**
* **对象属性通常是在 `__init__()` 里创建的。**

例如：
``` python
class MLP:

    def __init__(self):

        self.layer1 = nn.Linear(3,4)
```

这里 self.layer1 是在运行时 动态创建对象属性。

##### 2.2 **❓Python 中如果像 Java 一样写会发生什么？**

如果你写：
``` python
class MLP(nn.Module):

    layer1 = nn.Linear(3,4)
```

这其实会变成：

`类属性（class attribute）`

而不是：

`对象属性（instance attribute）`

**类似于java中的 static 静态变量**

这会导致严重问题：
* 所有对象共享同一个 layer
* 参数不会正确注册
* PyTorch Module 机制会混乱

**所以 PyTorch 明确要求在 `__init__` 中用 self 定义层**

##### 2.3 **❓为什么 PyTorch 必须用 self？**

因为 nn.Module 在内部做了一个非常关键的事情：

当你写：

`self.layer1 = nn.Linear(3,4)`

PyTorch 会自动执行：

`register_module("layer1", layer1)`

这一步会把这个层注册到模型里。

注册后 PyTorch 才能：
* 找到参数
* 自动计算梯度
* optimizer 更新参数
* 保存 / 加载模型

如果没有 self：这一步不会发生。

#### 3. 哪些参数应该定义在模型内部?
🧠 在 PyTorch 中，哪些东西应该写在模型内部，哪些应该写在模型外部？

##### 3.1 哪些内容应该在模型内部定义？
在 `__init__()` 中，我们通常会定义：
* Linear 层
* Conv 层
* 激活函数
* Dropout
* BatchNorm

例如：
`Linear → ReLU → Linear`

这些组件共同构成：**神经网络结构（Model Architecture）**

也就是说，它们决定了：
* 网络有多少层
* 每一层做什么计算
* 数据如何在网络中流动

因此：

**🧠 凡是属于模型结构的内容，都必须定义在模型内部。**

##### 3.2 为什么 forward() 必须写在模型内部？
forward() 的作用是定义：
`输入数据如何通过网络得到输出`

例如：

`x → layer1 → relu → layer2 → output`

也就是说，forward() 描述的是：
* 数据流动路径
* 每一层的计算顺序

因此 forward() 实际上定义的是：

`模型的计算图（computational graph）`

**❓为什么必须在模型内部？**
因为 forward() 需要访问模型中的层，例如：
```
self.layer1
self.relu
self.layer2
```
如果 forward() 不在模型内部，就无法访问这些层。

因此：

**🧠 模型结构在 `__init__` 中定义，数据计算逻辑在 forward() 中定义。**

##### 3.3 为什么 model(X) 会自动执行 forward()？
PyTorch 的模型调用机制是：

当你写：

`y_pred = model(X)`

实际上发生的是：

`model.__call__(X)`

而 __call__() 内部会自动执行：

`forward(X)`

因此：

`model(X)`

本质上等价于：

`model.forward(X)`

但是 PyTorch 推荐使用：
`model(X)`

因为这样可以：
* 自动构建计算图
* 自动支持 hooks
* 自动支持 autograd

#### 4. 哪些参数应该定义在模型外部

##### 4.1 为什么损失函数不在模型内部？
损失函数（Loss）的作用是：

`衡量模型预测结果与真实标签之间的差距`

例如：

`loss = criterion(y_pred, y_true)`

损失函数涉及两个东西：
* 模型输出 y_pred
* 真实标签 y_true

因此它属于：
* 模型训练过程中的一部分
* 而不是模型结构的一部分

所以：
🧠 损失函数应该在模型外部定义。

##### 4.2 为什么优化器也在模型外部？
优化器（Optimizer）的作用是：

`根据梯度更新模型参数`

例如：
`optimizer = Adam(model.parameters())`

优化器负责：
* 梯度下降
* 学习率控制
* 参数更新策略

这些属于：
* 训练算法（training algorithm）
* 而不是模型结构。

因此：**🧠 优化器也应该定义在模型外部。**

#### 5. 模型搭建总结🧠
💡 PyTorch 模型设计的核心逻辑
我们可以把 PyTorch 的代码结构分为 三部分：

##### 5.1 模型结构（Model Architecture）
定义在：
`__init__()`

例如：
* Linear
* Conv
* ReLU
* Dropout

##### 5.2 数据计算流程（Forward Pass）
定义在：

`forward()`

描述：

`输入 → layer → activation → layer → 输出`

##### 5.3 训练策略（Training Strategy）
定义在 **模型外部**

例如：
* Loss
* Optimizer
* Training Loop

```
凡是用于得到预测值的步骤，都应该放在模型内部定义
根据预测值计算得到损失函数值，根据反向传播计算梯度和更新梯度的操作-
属于模型的训练策略，不属于模型结构本身，应该放在模型外部定义
```